In [ ]:
from datasets import load_dataset

marwari = load_dataset(
    "ARTPARK-IISc/Vaani-transcription-part",
    "Marwari",
    split="test",
    token="token",   # or omit if you ran huggingface-cli login
)

print(marwari)          # shows splits + feature schema
print(marwari["train"][0])   # inspect one example directly

In [ ]:
from datasets import load_dataset
# data ARTPARK-IISc/Vaani-transcription-part"
marwari = load_dataset(
    "parquet",
    data_files="/content/drive/MyDrive/test-00000-of-00001.parquet"
)


Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:

print(marwari)


DatasetDict({
    train: Dataset({
        features: ['audio', 'language', 'gender', 'state', 'district', 'transcript', 'referenceImage'],
        num_rows: 417
    })
})


In [ ]:
print(marwari["train"][0]["audio"])


In [ ]:
!git clone https://huggingface.co/facebook/wav2vec2-xls-r-300m

Cloning into 'wav2vec2-xls-r-300m'...
remote: Enumerating objects: 77, done.
remote: Total 77 (delta 0), reused 0 (delta 0), pack-reused 77 (from 1)
Receiving objects: 100% (77/77), 10.39 KiB | 10.39 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [ ]:
 """
Zero-Shot XLS-R Encoder Similarity — Bhili vs a single candidate donor
======================================================================


The transcript column is never read — audio only.

Layer zones (XLS-R-300M, 25 hidden states):
    L0 -> CNN | L1-6 -> Acoustic | L7-12 -> Phonetic (key) | L13-24 -> Linguistic

"""

import os, random, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import cosine
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION  <- edit this block only
# =============================================================================

CONFIG = {
    "base_encoder": "wav2vec2-xls-r-300m",

    # Reference language (Bhili): a folder of wav/flac/mp3, searched recursively.
    "bhili_audio_dir": "bhili_audio",  # or "bhili_audio_subset" for a smaller test set

    # Candidate donor: a saved HF DatasetDict dir (load_from_disk) with an
    # "audio" column. Set the label to whatever the data actually is.
    "candidate_label": "Mewari",                       # or "Marwari"
    "candidate_hf_path": "test-00000-of-00001.parquet",

    "paper_reference_points": {"Hindi (accepted)": 0.751,
                               "Gujarati (rejected)": 0.067},

    "num_samples":    417,      # candidate has 417 rows -> all used
    "num_layers":     25,
    "output_dir":     "bhili_candidate_outputs",
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "sample_rate":    16000,
    "max_duration_s": 10,
    "seed":           42,
}

BG = "#0F1117"
ZONES = [
    (0,  1,  "#3A86FF", "CNN Features (L0)"),
    (1,  7,  "#8338EC", "Acoustic (L1-6)"),
    (7,  13, "#FF006E", "Phonetic (L7-12)"),
    (13, 25, "#FB5607", "Linguistic (L13-24)"),
]
CAND_COLOR = "#6A4C93"


# =============================================================================
# Audio loading (audio only; transcript ignored)
# =============================================================================

def _finalize(arr, cfg):
    max_len = int(cfg["max_duration_s"] * cfg["sample_rate"])
    arr = np.asarray(arr, dtype=np.float32)[:max_len]
    if len(arr) < 400:
        return None
    peak = np.abs(arr).max()
    if peak > 0:
        arr = arr / peak
    return arr


def load_bhili_dir(cfg):
    import glob, librosa
    exts = ("*.wav", "*.flac", "*.mp3", "*.ogg")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(cfg["bhili_audio_dir"], "**", ext),
                               recursive=True))
    files = sorted(files)
    if not files:
        raise FileNotFoundError(f"No audio under '{cfg['bhili_audio_dir']}'.")
    random.seed(cfg["seed"])
    if len(files) > cfg["num_samples"]:
        files = random.sample(files, cfg["num_samples"])
    out, skipped = [], 0
    for fp in tqdm(files, desc="  Bhili audio", leave=False):
        try:
            wav, _ = librosa.load(fp, sr=cfg["sample_rate"], mono=True)
            wav = _finalize(wav, cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  Bhili: loaded {len(out)} clips ({skipped} skipped)")
    return out


def load_candidate_hf(cfg):
    from datasets import load_dataset, Audio
    ds = load_dataset("parquet", data_files=cfg["candidate_hf_path"])
    # DatasetDict -> take the only split ("train" by default for a single file)
    ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
    ds = ds.cast_column("audio", Audio(sampling_rate=cfg["sample_rate"]))
    idxs = list(range(len(ds)))
    random.seed(cfg["seed"])
    if len(idxs) > cfg["num_samples"]:
        idxs = random.sample(idxs, cfg["num_samples"])
    out, skipped = [], 0
    for i in tqdm(idxs, desc=f"  {cfg['candidate_label']} audio", leave=False):
        try:
            wav = _finalize(ds[i]["audio"]["array"], cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {cfg['candidate_label']}: loaded {len(out)} clips ({skipped} skipped)")
    return out


# =============================================================================
# Embeddings + similarity
# =============================================================================

def extract_layerwise_embeddings(audio_list, cfg, fe, model):
    device = cfg["device"]
    layer_accum, count = None, 0
    for audio in tqdm(audio_list, desc="    Extracting", leave=False):
        try:
            inputs = fe(audio, sampling_rate=cfg["sample_rate"],
                        return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)
            clip = np.stack([hs.squeeze(0).mean(0).cpu().float().numpy()
                             for hs in out.hidden_states])
            layer_accum = clip if layer_accum is None else layer_accum + clip
            count += 1
        except Exception:
            pass
    if count == 0:
        raise RuntimeError("No embeddings extracted.")
    return layer_accum / count


def compute_layerwise_cosine(ref, other):
    return [float(1 - cosine(ref[i], other[i])) for i in range(ref.shape[0])]


# =============================================================================
# Plots + summary
# =============================================================================

def plot_line(sims, cfg):
    os.makedirs(cfg["output_dir"], exist_ok=True)
    layers = list(range(cfg["num_layers"]))
    labels = ["CNN"] + [str(i) for i in range(1, cfg["num_layers"])]
    fig, ax = plt.subplots(figsize=(15, 6))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for s, e, c, _ in ZONES:
        ax.axvspan(s - 0.5, e - 0.5, alpha=0.09, color=c)
    ax.plot(layers, sims, color=CAND_COLOR, linewidth=2.5, marker="D",
            markersize=4, label=cfg["candidate_label"])
    pl = 7 + int(np.argmax(sims[7:13])); pv = max(sims[7:13])
    ax.annotate(f"  {pv:.3f}", xy=(pl, pv), color=CAND_COLOR, fontsize=9,
                fontweight="bold")
    ax.set_xlabel("Layer Index", color="#CCCCCC", fontsize=12)
    ax.set_ylabel("Cosine Similarity with Bhili", color="#CCCCCC", fontsize=12)
    ax.set_title(f"Zero-shot XLS-R Layer-wise Similarity: Bhili vs {cfg['candidate_label']}\n"
                 "(base XLS-R-300M encoder)", color="white", fontsize=14,
                 fontweight="bold", pad=15)
    ax.set_xticks(layers); ax.set_xticklabels(labels, rotation=45,
                                              color="#AAAAAA", fontsize=8)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, alpha=0.15, color="#FFFFFF", linestyle="--")
    lh = [plt.Line2D([0], [0], color=CAND_COLOR, marker="D", linewidth=2.5,
                     label=cfg["candidate_label"])]
    zh = [mpatches.Patch(color=c, alpha=0.5, label=lb) for _, _, c, lb in ZONES]
    ax.legend(handles=lh + zh, loc="upper right", fontsize=9,
              facecolor="#1A1A2E", edgecolor="#444444", labelcolor="white")
    plt.tight_layout()
    p = f"{cfg['output_dir']}/bhili_vs_{cfg['candidate_label'].lower()}_layerwise.png"
    plt.savefig(p, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved -> {p}")
    return p


def summary(sims, cfg):
    zs = {"CNN (L0)": slice(0, 1),
          "Acoustic (L1-6)": slice(1, 7),
          "Phonetic (L7-12)": slice(7, 13),
          "Linguistic (L13-24)": slice(13, 25),
          "Overall mean": slice(0, cfg["num_layers"])}
    row = {"Language": cfg["candidate_label"]}
    for name, s in zs.items():
        row[name] = round(float(np.mean(sims[s])), 4)
    row["Peak layer"] = int(np.argmax(sims))
    row["Peak value"] = round(float(np.max(sims)), 4)
    df = pd.DataFrame([row]).set_index("Language")
    sep = "=" * 70
    print(f"\n{sep}\n  ZERO-SHOT XLS-R SIMILARITY WITH BHILI (base encoder)\n{sep}")
    print(df.to_string()); print(sep)
    os.makedirs(cfg["output_dir"], exist_ok=True)
    csv = f"{cfg['output_dir']}/bhili_vs_{cfg['candidate_label'].lower()}_summary.csv"
    df.to_csv(csv); print(f"  Summary CSV -> {csv}")
    return df, float(np.mean(sims[7:13]))


# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    print(f"\nDevice        : {cfg['device']}")
    print(f"Base encoder  : {cfg['base_encoder']}")
    print(f"Reference     : Bhili")
    print(f"Candidate     : {cfg['candidate_label']}")

    print("\n[1/4] Loading base XLS-R-300M encoder ...")
    fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg["base_encoder"])
    model = Wav2Vec2Model.from_pretrained(cfg["base_encoder"])
    model.eval().to(cfg["device"])

    print("\n[2/4] Loading audio + extracting embeddings ...")
    bhili_audio = load_bhili_dir(cfg)
    cand_audio = load_candidate_hf(cfg)
    bhili_emb = extract_layerwise_embeddings(bhili_audio, cfg, fe, model)
    cand_emb = extract_layerwise_embeddings(cand_audio, cfg, fe, model)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    sims = compute_layerwise_cosine(bhili_emb, cand_emb)

    print("\n[3/4] Plot")
    plot_line(sims, cfg)

    print("\n[4/4] Summary")
    df, phon = summary(sims, cfg)

    print(f"\n  {cfg['candidate_label']} phonetic-zone (L7-12) similarity with Bhili: {phon:.4f}")
    print("  Paper reference points (Table 8/9, donor-adapted):")
    for k, v in cfg["paper_reference_points"].items():
        print(f"    {k:22s}: {v:.3f}")
    guj = cfg["paper_reference_points"].get("Gujarati (rejected)")
    if guj is not None:
        verdict = "at or below" if phon <= guj + 0.05 else "above"
        print(f"\n  -> {cfg['candidate_label']} phonetic similarity is {verdict} "
              f"the rejected Gujarati donor.")
    print(f"\n  Done. Outputs in: {cfg['output_dir']}/\n")


if __name__ == "__main__":
    main()

In [ ]:
 """
Zero-Shot XLS-R Encoder Similarity — Bhili vs a single candidate donor
======================================================================

"""

import os, random, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import cosine
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION  <- edit this block only
# =============================================================================

CONFIG = {
    "base_encoder": "/wav2vec2-xls-r-300m",

    # Reference language (Bhili): a folder of wav/flac/mp3, searched recursively.
    "bhili_audio_dir": "bhili_audio",  # or "bhili_audio_subset" for a smaller test set

    # Candidate donor: a saved HF DatasetDict dir (load_from_disk) with an
    # "audio" column. Set the label to whatever the data actually is.
    "candidate_label": "Mewari",                       # or "Marwari"
    "candidate_hf_path": "validation-00000-of-00001.parquet",

    # Paper's published phonetic-zone reference points (for interpretation only).
    "paper_reference_points": {"Hindi (accepted)": 0.751,
                               "Gujarati (rejected)": 0.067},

    "num_samples":    417,      # candidate has 417 rows -> all used
    "num_layers":     25,
    "output_dir":     "bhili_candidate_outputs",
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "sample_rate":    16000,
    "max_duration_s": 10,
    "seed":           42,
}

BG = "#0F1117"
ZONES = [
    (0,  1,  "#3A86FF", "CNN Features (L0)"),
    (1,  7,  "#8338EC", "Acoustic (L1-6)"),
    (7,  13, "#FF006E", "Phonetic (L7-12)"),
    (13, 25, "#FB5607", "Linguistic (L13-24)"),
]
CAND_COLOR = "#6A4C93"


# =============================================================================
# Audio loading (audio only; transcript ignored)
# =============================================================================

def _finalize(arr, cfg):
    max_len = int(cfg["max_duration_s"] * cfg["sample_rate"])
    arr = np.asarray(arr, dtype=np.float32)[:max_len]
    if len(arr) < 400:
        return None
    peak = np.abs(arr).max()
    if peak > 0:
        arr = arr / peak
    return arr


def load_bhili_dir(cfg):
    import glob, librosa
    exts = ("*.wav", "*.flac", "*.mp3", "*.ogg")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(cfg["bhili_audio_dir"], "**", ext),
                               recursive=True))
    files = sorted(files)
    if not files:
        raise FileNotFoundError(f"No audio under '{cfg['bhili_audio_dir']}'.")
    random.seed(cfg["seed"])
    if len(files) > cfg["num_samples"]:
        files = random.sample(files, cfg["num_samples"])
    out, skipped = [], 0
    for fp in tqdm(files, desc="  Bhili audio", leave=False):
        try:
            wav, _ = librosa.load(fp, sr=cfg["sample_rate"], mono=True)
            wav = _finalize(wav, cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  Bhili: loaded {len(out)} clips ({skipped} skipped)")
    return out


def load_candidate_hf(cfg):
    from datasets import load_dataset, Audio
    ds = load_dataset("parquet", data_files=cfg["candidate_hf_path"])
    # DatasetDict -> take the only split ("train" by default for a single file)
    ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
    ds = ds.cast_column("audio", Audio(sampling_rate=cfg["sample_rate"]))
    idxs = list(range(len(ds)))
    random.seed(cfg["seed"])
    if len(idxs) > cfg["num_samples"]:
        idxs = random.sample(idxs, cfg["num_samples"])
    out, skipped = [], 0
    for i in tqdm(idxs, desc=f"  {cfg['candidate_label']} audio", leave=False):
        try:
            wav = _finalize(ds[i]["audio"]["array"], cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {cfg['candidate_label']}: loaded {len(out)} clips ({skipped} skipped)")
    return out


# =============================================================================
# Embeddings + similarity
# =============================================================================

def extract_layerwise_embeddings(audio_list, cfg, fe, model):
    device = cfg["device"]
    layer_accum, count = None, 0
    for audio in tqdm(audio_list, desc="    Extracting", leave=False):
        try:
            inputs = fe(audio, sampling_rate=cfg["sample_rate"],
                        return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)
            clip = np.stack([hs.squeeze(0).mean(0).cpu().float().numpy()
                             for hs in out.hidden_states])
            layer_accum = clip if layer_accum is None else layer_accum + clip
            count += 1
        except Exception:
            pass
    if count == 0:
        raise RuntimeError("No embeddings extracted.")
    return layer_accum / count


def compute_layerwise_cosine(ref, other):
    return [float(1 - cosine(ref[i], other[i])) for i in range(ref.shape[0])]


# =============================================================================
# Plots + summary
# =============================================================================

def plot_line(sims, cfg):
    os.makedirs(cfg["output_dir"], exist_ok=True)
    layers = list(range(cfg["num_layers"]))
    labels = ["CNN"] + [str(i) for i in range(1, cfg["num_layers"])]
    fig, ax = plt.subplots(figsize=(15, 6))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for s, e, c, _ in ZONES:
        ax.axvspan(s - 0.5, e - 0.5, alpha=0.09, color=c)
    ax.plot(layers, sims, color=CAND_COLOR, linewidth=2.5, marker="D",
            markersize=4, label=cfg["candidate_label"])
    pl = 7 + int(np.argmax(sims[7:13])); pv = max(sims[7:13])
    ax.annotate(f"  {pv:.3f}", xy=(pl, pv), color=CAND_COLOR, fontsize=9,
                fontweight="bold")
    ax.set_xlabel("Layer Index", color="#CCCCCC", fontsize=12)
    ax.set_ylabel("Cosine Similarity with Bhili", color="#CCCCCC", fontsize=12)
    ax.set_title(f"Zero-shot XLS-R Layer-wise Similarity: Bhili vs {cfg['candidate_label']}\n"
                 "(base XLS-R-300M encoder)", color="white", fontsize=14,
                 fontweight="bold", pad=15)
    ax.set_xticks(layers); ax.set_xticklabels(labels, rotation=45,
                                              color="#AAAAAA", fontsize=8)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, alpha=0.15, color="#FFFFFF", linestyle="--")
    lh = [plt.Line2D([0], [0], color=CAND_COLOR, marker="D", linewidth=2.5,
                     label=cfg["candidate_label"])]
    zh = [mpatches.Patch(color=c, alpha=0.5, label=lb) for _, _, c, lb in ZONES]
    ax.legend(handles=lh + zh, loc="upper right", fontsize=9,
              facecolor="#1A1A2E", edgecolor="#444444", labelcolor="white")
    plt.tight_layout()
    p = f"{cfg['output_dir']}/bhili_vs_{cfg['candidate_label'].lower()}_layerwise.png"
    plt.savefig(p, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved -> {p}")
    return p


def summary(sims, cfg):
    zs = {"CNN (L0)": slice(0, 1),
          "Acoustic (L1-6)": slice(1, 7),
          "Phonetic (L7-12)": slice(7, 13),
          "Linguistic (L13-24)": slice(13, 25),
          "Overall mean": slice(0, cfg["num_layers"])}
    row = {"Language": cfg["candidate_label"]}
    for name, s in zs.items():
        row[name] = round(float(np.mean(sims[s])), 4)
    row["Peak layer"] = int(np.argmax(sims))
    row["Peak value"] = round(float(np.max(sims)), 4)
    df = pd.DataFrame([row]).set_index("Language")
    sep = "=" * 70
    print(f"\n{sep}\n  ZERO-SHOT XLS-R SIMILARITY WITH BHILI (base encoder)\n{sep}")
    print(df.to_string()); print(sep)
    os.makedirs(cfg["output_dir"], exist_ok=True)
    csv = f"{cfg['output_dir']}/bhili_vs_{cfg['candidate_label'].lower()}_summary.csv"
    df.to_csv(csv); print(f"  Summary CSV -> {csv}")
    return df, float(np.mean(sims[7:13]))


# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    print(f"\nDevice        : {cfg['device']}")
    print(f"Base encoder  : {cfg['base_encoder']}")
    print(f"Reference     : Bhili")
    print(f"Candidate     : {cfg['candidate_label']}")

    print("\n[1/4] Loading base XLS-R-300M encoder ...")
    fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg["base_encoder"])
    model = Wav2Vec2Model.from_pretrained(cfg["base_encoder"])
    model.eval().to(cfg["device"])

    print("\n[2/4] Loading audio + extracting embeddings ...")
    bhili_audio = load_bhili_dir(cfg)
    cand_audio = load_candidate_hf(cfg)
    bhili_emb = extract_layerwise_embeddings(bhili_audio, cfg, fe, model)
    cand_emb = extract_layerwise_embeddings(cand_audio, cfg, fe, model)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    sims = compute_layerwise_cosine(bhili_emb, cand_emb)

    print("\n[3/4] Plot")
    plot_line(sims, cfg)

    print("\n[4/4] Summary")
    df, phon = summary(sims, cfg)

    print(f"\n  {cfg['candidate_label']} phonetic-zone (L7-12) similarity with Bhili: {phon:.4f}")
    print("  Paper reference points (Table 8/9, donor-adapted):")
    for k, v in cfg["paper_reference_points"].items():
        print(f"    {k:22s}: {v:.3f}")
    guj = cfg["paper_reference_points"].get("Gujarati (rejected)")
    if guj is not None:
        verdict = "at or below" if phon <= guj + 0.05 else "above"
        print(f"\n  -> {cfg['candidate_label']} phonetic similarity is {verdict} "
              f"the rejected Gujarati donor.")
    print(f"\n  Done. Outputs in: {cfg['output_dir']}/\n")


if __name__ == "__main__":
    main()

In [ ]:
"""
Candidate-Donor Similarity in Bhili Fine-Tuned Space — XLSR-300M
================================================================



Layer zones (XLSR-300M, 25 hidden states):
    L0 -> CNN | L1-6 -> Acoustic | L7-12 -> Phonetic (key) | L13-24 -> Linguistic

Requirements:
    pip install transformers torch librosa datasets scipy matplotlib seaborn pandas numpy tqdm
"""

import os, glob, random, warnings
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import cosine
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION  <- edit this block only
# =============================================================================

CONFIG = {
    # The ONE fine-tuned model all audio is encoded through.
    "bhili_model": "model",

    # Audio sources. Each entry is either:
    #   {"type": "dir",     "path": <folder of wav/flac/mp3>}
    #   {"type": "parquet", "path": <.parquet file with an "audio" column>}
    # "Bhili" is the reference; every other language is compared against it.
    "audio_sources": {
        "Bhili":    {"type": "dir",     "path": "bhili_audio"},
        "Mewari":   {"type": "parquet", "path": "mewari.parquet"},
        "Gujarati": {"type": "parquet", "path": "gujarati.parquet"},
    },
    "reference": "Bhili",

    # Paper's published phonetic-zone value for the rejected Gujarati donor,
    # printed for cross-reference (Table 8/9, donor-adapted pipeline).
    "paper_gujarati_phonetic": 0.067,
    
    "num_samples":    417,      # cap per language (Mewari has 417 -> all used)
    "num_layers":     25,
    "output_dir":     "mewari_bhilispace_outputs",
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "sample_rate":    16000,
    "max_duration_s": 10,
    "seed":           42,
}

LANG_COLORS = {"Mewari": "#6A4C93", "Gujarati": "#E63946"}
LANG_STYLES = {
    "Mewari":   {"linestyle": "--", "linewidth": 2.5, "marker": "D", "markersize": 4},
    "Gujarati": {"linestyle": "-",  "linewidth": 2.5, "marker": "o", "markersize": 4},
}
_FC = ["#F4A261", "#A8DADC", "#457B9D", "#CDB4DB"]
_FS = [{"linestyle": "-.", "linewidth": 2.5, "marker": "s", "markersize": 4}]
BG  = "#0F1117"
ZONES = [
    (0,  1,  "#3A86FF", "CNN Features (L0)"),
    (1,  7,  "#8338EC", "Acoustic (L1-6)"),
    (7,  13, "#FF006E", "Phonetic (L7-12)"),
    (13, 25, "#FB5607", "Linguistic (L13-24)"),
]

def _color(lang, idx): return LANG_COLORS.get(lang, _FC[idx % len(_FC)])
def _style(lang, idx): return LANG_STYLES.get(lang, _FS[idx % len(_FS)])


# =============================================================================
# Audio loading (audio only; transcript ignored)
# =============================================================================

def _finalize(arr, cfg):
    max_len = int(cfg["max_duration_s"] * cfg["sample_rate"])
    arr = np.asarray(arr, dtype=np.float32)[:max_len]
    if len(arr) < 400:
        return None
    peak = np.abs(arr).max()
    if peak > 0:
        arr = arr / peak
    return arr


def _load_dir(path, cfg, label):
    exts = ("*.wav", "*.flac", "*.mp3", "*.ogg")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(path, "**", ext), recursive=True))
    files = sorted(files)
    if not files:
        raise FileNotFoundError(f"No audio under '{path}' for {label}.")
    random.seed(cfg["seed"])
    if len(files) > cfg["num_samples"]:
        files = random.sample(files, cfg["num_samples"])
    out, skipped = [], 0
    for fp in tqdm(files, desc=f"  {label} (dir)", leave=False):
        try:
            wav, _ = librosa.load(fp, sr=cfg["sample_rate"], mono=True)
            wav = _finalize(wav, cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {label}: loaded {len(out)} clips ({skipped} skipped)")
    return out


def _load_parquet(path, cfg, label):
    from datasets import load_dataset, Audio
    ds = load_dataset("parquet", data_files=path)
    ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
    ds = ds.cast_column("audio", Audio(sampling_rate=cfg["sample_rate"]))
    idxs = list(range(len(ds)))
    random.seed(cfg["seed"])
    if len(idxs) > cfg["num_samples"]:
        idxs = random.sample(idxs, cfg["num_samples"])
    out, skipped = [], 0
    for i in tqdm(idxs, desc=f"  {label} (parquet)", leave=False):
        try:
            wav = _finalize(ds[i]["audio"]["array"], cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {label}: loaded {len(out)} clips ({skipped} skipped)")
    return out


def load_audio(label, spec, cfg):
    if spec["type"] == "dir":
        return _load_dir(spec["path"], cfg, label)
    elif spec["type"] == "parquet":
        return _load_parquet(spec["path"], cfg, label)
    raise ValueError(f"Unknown source type: {spec['type']}")


# =============================================================================
# Embeddings (all audio through the ONE Bhili model) + similarity
# =============================================================================

def extract_layerwise_embeddings(audio_list, cfg, fe, model, label):
    device = cfg["device"]
    layer_accum, count = None, 0
    for audio in tqdm(audio_list, desc=f"    Encoding {label}", leave=False):
        try:
            inputs = fe(audio, sampling_rate=cfg["sample_rate"],
                        return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)
            clip = np.stack([hs.squeeze(0).mean(0).cpu().float().numpy()
                             for hs in out.hidden_states])
            layer_accum = clip if layer_accum is None else layer_accum + clip
            count += 1
        except Exception:
            pass
    if count == 0:
        raise RuntimeError(f"No embeddings extracted for {label}.")
    return layer_accum / count


def compute_layerwise_cosine(ref, other):
    return [float(1 - cosine(ref[i], other[i])) for i in range(ref.shape[0])]


# =============================================================================
# Plots + summary
# =============================================================================

def plot_all(similarity_dict, cfg):
    os.makedirs(cfg["output_dir"], exist_ok=True)
    layers = list(range(cfg["num_layers"]))
    labels = ["CNN"] + [str(i) for i in range(1, cfg["num_layers"])]

    # line
    fig, ax = plt.subplots(figsize=(15, 6))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for s, e, c, _ in ZONES:
        ax.axvspan(s - 0.5, e - 0.5, alpha=0.09, color=c)
    for idx, (lang, sims) in enumerate(similarity_dict.items()):
        ax.plot(layers, sims, color=_color(lang, idx), label=lang, **_style(lang, idx))
        pl = 7 + int(np.argmax(sims[7:13])); pv = max(sims[7:13])
        ax.annotate(f"  {pv:.3f}", xy=(pl, pv), color=_color(lang, idx),
                    fontsize=8, fontweight="bold")
    ax.set_xlabel("Layer Index", color="#CCCCCC", fontsize=12)
    ax.set_ylabel("Cosine Similarity with Bhili (in Bhili model space)",
                  color="#CCCCCC", fontsize=12)
    ax.set_title("Candidate-Donor Similarity in Bhili Fine-Tuned Space\n"
                 "(Bhili / Mewari / Gujarati audio encoded by the Bhili model)",
                 color="white", fontsize=14, fontweight="bold", pad=15)
    ax.set_xticks(layers); ax.set_xticklabels(labels, rotation=45,
                                              color="#AAAAAA", fontsize=8)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, alpha=0.15, color="#FFFFFF", linestyle="--")
    lh = [plt.Line2D([0], [0], color=_color(l, i), label=l, **_style(l, i))
          for i, l in enumerate(similarity_dict)]
    zh = [mpatches.Patch(color=c, alpha=0.5, label=lb) for _, _, c, lb in ZONES]
    ax.legend(handles=lh + zh, loc="upper right", fontsize=9,
              facecolor="#1A1A2E", edgecolor="#444444", labelcolor="white")
    plt.tight_layout()
    p1 = f"{cfg['output_dir']}/layerwise_similarity.png"
    plt.savefig(p1, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved -> {p1}")

    # heatmap
    df_h = pd.DataFrame(similarity_dict, index=labels)
    fig, ax = plt.subplots(figsize=(max(6, len(similarity_dict) * 3), 12))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    sns.heatmap(df_h, ax=ax, cmap="RdYlGn",
                vmin=round(df_h.values.min() - 0.02, 2), vmax=1.0,
                annot=True, fmt=".3f", annot_kws={"size": 7},
                linewidths=0.4, linecolor="#222222",
                cbar_kws={"shrink": 0.6, "label": "Cosine Similarity"})
    ax.set_title("Similarity Heatmap (layer x language vs. Bhili)",
                 color="white", fontsize=12, fontweight="bold", pad=12)
    ax.tick_params(colors="#CCCCCC", labelsize=8)
    ax.set_xlabel("Language (audio)", color="#CCCCCC", fontsize=10)
    ax.set_ylabel("Layer", color="#CCCCCC", fontsize=10)
    cb = ax.collections[0].colorbar
    cb.ax.yaxis.label.set_color("white"); cb.ax.tick_params(colors="white")
    plt.tight_layout()
    p2 = f"{cfg['output_dir']}/similarity_heatmap.png"
    plt.savefig(p2, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved -> {p2}")

    # zone bars
    zs = {"Acoustic\n(L1-6)": slice(1, 7),
          "Phonetic\n(L7-12)": slice(7, 13),
          "Linguistic\n(L13-24)": slice(13, 25)}
    zn = list(zs.keys()); langs = list(similarity_dict.keys())
    nl = len(langs); tw = 0.7; bw = tw / nl
    offs = np.linspace(-tw / 2 + bw / 2, tw / 2 - bw / 2, nl)
    x = np.arange(len(zn))
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for idx, lang in enumerate(langs):
        sims = similarity_dict[lang]
        vals = [float(np.mean(sims[s])) for s in zs.values()]
        bars = ax.bar(x + offs[idx], vals, bw, label=lang,
                      color=_color(lang, idx), alpha=0.85, edgecolor="#222222")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                    f"{val:.3f}", ha="center", va="bottom",
                    color="white", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(zn, color="#CCCCCC", fontsize=11)
    ax.set_ylabel("Mean Cosine Similarity", color="#CCCCCC", fontsize=11)
    ax.set_title("Zone-averaged Similarity with Bhili (Bhili model space)",
                 color="white", fontsize=13, fontweight="bold", pad=12)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, axis="y", alpha=0.15, color="#FFFFFF", linestyle="--")
    ax.set_ylim(0, 1.08)
    ax.legend(facecolor="#1A1A2E", edgecolor="#444444", labelcolor="white", fontsize=10)
    plt.tight_layout()
    p3 = f"{cfg['output_dir']}/zone_similarity_bars.png"
    plt.savefig(p3, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved -> {p3}")
    return p1, p2, p3


def summary(similarity_dict, cfg):
    zs = {"CNN (L0)": slice(0, 1),
          "Acoustic (L1-6)": slice(1, 7),
          "Phonetic (L7-12)": slice(7, 13),
          "Linguistic (L13-24)": slice(13, 25),
          "Overall mean": slice(0, cfg["num_layers"])}
    rows = []
    for lang, sims in similarity_dict.items():
        row = {"Language": lang}
        for name, s in zs.items():
            row[name] = round(float(np.mean(sims[s])), 4)
        row["Peak layer"] = int(np.argmax(sims))
        row["Peak value"] = round(float(np.max(sims)), 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Language")
    sep = "=" * 78
    print(f"\n{sep}\n  SIMILARITY WITH BHILI IN BHILI FINE-TUNED SPACE\n{sep}")
    print(df.to_string()); print(sep)
    os.makedirs(cfg["output_dir"], exist_ok=True)
    csv = f"{cfg['output_dir']}/similarity_summary.csv"
    df.to_csv(csv); print(f"  Summary CSV -> {csv}")
    return df


# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    print(f"\nDevice       : {cfg['device']}")
    print(f"Bhili model  : {cfg['bhili_model']}")
    print(f"Audio langs  : {list(cfg['audio_sources'].keys())}")
    print(f"Reference    : {cfg['reference']}")

    print("\n[1/4] Loading Bhili fine-tuned model (shared encoder for all audio) ...")
    fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg["bhili_model"])
    model = Wav2Vec2Model.from_pretrained(cfg["bhili_model"])
    model.eval().to(cfg["device"])

    print("\n[2/4] Loading audio + encoding each language through the Bhili model ...")
    emb = {}
    for label, spec in cfg["audio_sources"].items():
        audio = load_audio(label, spec, cfg)
        emb[label] = extract_layerwise_embeddings(audio, cfg, fe, model, label)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    ref = cfg["reference"]
    similarity_dict = {}
    for label in cfg["audio_sources"]:
        if label == ref:
            continue
        sims = compute_layerwise_cosine(emb[ref], emb[label])
        similarity_dict[label] = sims
        print(f"     {label:9s} phonetic-zone mean (L7-12): {np.mean(sims[7:13]):.4f}")

    print("\n[3/4] Plots")
    plot_all(similarity_dict, cfg)

    print("\n[4/4] Summary")
    summary(similarity_dict, cfg)

    # Verdict
    phon = {l: float(np.mean(s[7:13])) for l, s in similarity_dict.items()}
    print("\n  Phonetic-zone (L7-12) similarity with Bhili (Bhili model space):")
    for l, v in sorted(phon.items(), key=lambda kv: -kv[1]):
        print(f"    {l:9s}: {v:.4f}")
    if "Mewari" in phon and "Gujarati" in phon:
        if phon["Mewari"] <= phon["Gujarati"] + 0.02:
            print(f"\n  -> Mewari ({phon['Mewari']:.4f}) sits at or below Gujarati "
                  f"({phon['Gujarati']:.4f}), the rejected donor. Supports exclusion.")
        else:
            print(f"\n  -> Mewari ({phon['Mewari']:.4f}) is ABOVE Gujarati "
                  f"({phon['Gujarati']:.4f}); interpret with care before using in rebuttal.")
    print(f"\n  (Paper's donor-adapted Gujarati phonetic value for reference: "
          f"{cfg['paper_gujarati_phonetic']})")
    print(f"\n  Done. Outputs in: {cfg['output_dir']}/\n")


if __name__ == "__main__":
    main()

In [ ]:
"""
Candidate-Donor Similarity in Bhili Fine-Tuned Space — XLSR-300M
================================================================

Layer zones (XLSR-300M, 25 hidden states):
    L0 -> CNN | L1-6 -> Acoustic | L7-12 -> Phonetic (key) | L13-24 -> Linguistic

Requirements:
    pip install transformers torch librosa datasets scipy matplotlib seaborn pandas numpy tqdm
"""

import os, glob, random, warnings
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import cosine
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION  <- edit this block only
# =============================================================================

CONFIG = {
    # The ONE fine-tuned model all audio is encoded through.
    "bhili_model": "model",

    # Audio sources. Each entry is either:
    #   {"type": "dir",     "path": <folder of wav/flac/mp3>}
    #   {"type": "parquet", "path": <.parquet file with an "audio" column>}
    # "Bhili" is the reference; every other language is compared against it.
    "audio_sources": {
        "Bhili":    {"type": "dir",     "path": "bhili_audio"},
        "Mewari":   {"type": "parquet", "path": "mewari.parquet"},
        "Gujarati": {"type": "parquet", "path": "gujarati.parquet"},
    },
    "reference": "Bhili",

    # Paper's published phonetic-zone value for the rejected Gujarati donor,
    # printed for cross-reference (Table 8/9, donor-adapted pipeline).
    "paper_gujarati_phonetic": 0.067,

    "num_samples":    417,      # cap per language (Mewari has 417 -> all used)
    "num_layers":     25,
    "output_dir":     "mewari_bhilispace_outputs",
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "sample_rate":    16000,
    "max_duration_s": 10,
    "seed":           42,
}

LANG_COLORS = {"Mewari": "#6A4C93", "Gujarati": "#E63946"}
LANG_STYLES = {
    "Mewari":   {"linestyle": "--", "linewidth": 2.5, "marker": "D", "markersize": 4},
    "Gujarati": {"linestyle": "-",  "linewidth": 2.5, "marker": "o", "markersize": 4},
}
_FC = ["#F4A261", "#A8DADC", "#457B9D", "#CDB4DB"]
_FS = [{"linestyle": "-.", "linewidth": 2.5, "marker": "s", "markersize": 4}]
BG  = "#0F1117"
ZONES = [
    (0,  1,  "#3A86FF", "CNN Features (L0)"),
    (1,  7,  "#8338EC", "Acoustic (L1-6)"),
    (7,  13, "#FF006E", "Phonetic (L7-12)"),
    (13, 25, "#FB5607", "Linguistic (L13-24)"),
]

def _color(lang, idx): return LANG_COLORS.get(lang, _FC[idx % len(_FC)])
def _style(lang, idx): return LANG_STYLES.get(lang, _FS[idx % len(_FS)])


# =============================================================================
# Audio loading (audio only; transcript ignored)
# =============================================================================

def _finalize(arr, cfg):
    max_len = int(cfg["max_duration_s"] * cfg["sample_rate"])
    arr = np.asarray(arr, dtype=np.float32)[:max_len]
    if len(arr) < 400:
        return None
    peak = np.abs(arr).max()
    if peak > 0:
        arr = arr / peak
    return arr


def _load_dir(path, cfg, label):
    exts = ("*.wav", "*.flac", "*.mp3", "*.ogg")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(path, "**", ext), recursive=True))
    files = sorted(files)
    if not files:
        raise FileNotFoundError(f"No audio under '{path}' for {label}.")
    random.seed(cfg["seed"])
    if len(files) > cfg["num_samples"]:
        files = random.sample(files, cfg["num_samples"])
    out, skipped = [], 0
    for fp in tqdm(files, desc=f"  {label} (dir)", leave=False):
        try:
            wav, _ = librosa.load(fp, sr=cfg["sample_rate"], mono=True)
            wav = _finalize(wav, cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {label}: loaded {len(out)} clips ({skipped} skipped)")
    return out


def _load_parquet(path, cfg, label):
    from datasets import load_dataset, Audio
    ds = load_dataset("parquet", data_files=path)
    ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
    ds = ds.cast_column("audio", Audio(sampling_rate=cfg["sample_rate"]))
    idxs = list(range(len(ds)))
    random.seed(cfg["seed"])
    if len(idxs) > cfg["num_samples"]:
        idxs = random.sample(idxs, cfg["num_samples"])
    out, skipped = [], 0
    for i in tqdm(idxs, desc=f"  {label} (parquet)", leave=False):
        try:
            wav = _finalize(ds[i]["audio"]["array"], cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {label}: loaded {len(out)} clips ({skipped} skipped)")
    return out


def load_audio(label, spec, cfg):
    if spec["type"] == "dir":
        return _load_dir(spec["path"], cfg, label)
    elif spec["type"] == "parquet":
        return _load_parquet(spec["path"], cfg, label)
    raise ValueError(f"Unknown source type: {spec['type']}")


# =============================================================================
# Embeddings (all audio through the ONE Bhili model) + similarity
# =============================================================================

def extract_layerwise_embeddings(audio_list, cfg, fe, model, label):
    device = cfg["device"]
    layer_accum, count = None, 0
    for audio in tqdm(audio_list, desc=f"    Encoding {label}", leave=False):
        try:
            inputs = fe(audio, sampling_rate=cfg["sample_rate"],
                        return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)
            clip = np.stack([hs.squeeze(0).mean(0).cpu().float().numpy()
                             for hs in out.hidden_states])
            layer_accum = clip if layer_accum is None else layer_accum + clip
            count += 1
        except Exception:
            pass
    if count == 0:
        raise RuntimeError(f"No embeddings extracted for {label}.")
    return layer_accum / count


def compute_layerwise_cosine(ref, other):
    return [float(1 - cosine(ref[i], other[i])) for i in range(ref.shape[0])]


# =============================================================================
# Plots + summary
# =============================================================================

def plot_all(similarity_dict, cfg):
    os.makedirs(cfg["output_dir"], exist_ok=True)
    layers = list(range(cfg["num_layers"]))
    labels = ["CNN"] + [str(i) for i in range(1, cfg["num_layers"])]

    # line
    fig, ax = plt.subplots(figsize=(15, 6))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for s, e, c, _ in ZONES:
        ax.axvspan(s - 0.5, e - 0.5, alpha=0.09, color=c)
    for idx, (lang, sims) in enumerate(similarity_dict.items()):
        ax.plot(layers, sims, color=_color(lang, idx), label=lang, **_style(lang, idx))
        pl = 7 + int(np.argmax(sims[7:13])); pv = max(sims[7:13])
        ax.annotate(f"  {pv:.3f}", xy=(pl, pv), color=_color(lang, idx),
                    fontsize=8, fontweight="bold")
    ax.set_xlabel("Layer Index", color="#CCCCCC", fontsize=12)
    ax.set_ylabel("Cosine Similarity with Bhili (in Bhili model space)",
                  color="#CCCCCC", fontsize=12)
    ax.set_title("Candidate-Donor Similarity in Bhili Fine-Tuned Space\n"
                 "(Bhili / Mewari / Gujarati audio encoded by the Bhili model)",
                 color="white", fontsize=14, fontweight="bold", pad=15)
    ax.set_xticks(layers); ax.set_xticklabels(labels, rotation=45,
                                              color="#AAAAAA", fontsize=8)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, alpha=0.15, color="#FFFFFF", linestyle="--")
    lh = [plt.Line2D([0], [0], color=_color(l, i), label=l, **_style(l, i))
          for i, l in enumerate(similarity_dict)]
    zh = [mpatches.Patch(color=c, alpha=0.5, label=lb) for _, _, c, lb in ZONES]
    ax.legend(handles=lh + zh, loc="upper right", fontsize=9,
              facecolor="#1A1A2E", edgecolor="#444444", labelcolor="white")
    plt.tight_layout()
    p1 = f"{cfg['output_dir']}/layerwise_similarity.png"
    plt.savefig(p1, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved -> {p1}")

    # heatmap
    df_h = pd.DataFrame(similarity_dict, index=labels)
    fig, ax = plt.subplots(figsize=(max(6, len(similarity_dict) * 3), 12))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    sns.heatmap(df_h, ax=ax, cmap="RdYlGn",
                vmin=round(df_h.values.min() - 0.02, 2), vmax=1.0,
                annot=True, fmt=".3f", annot_kws={"size": 7},
                linewidths=0.4, linecolor="#222222",
                cbar_kws={"shrink": 0.6, "label": "Cosine Similarity"})
    ax.set_title("Similarity Heatmap (layer x language vs. Bhili)",
                 color="white", fontsize=12, fontweight="bold", pad=12)
    ax.tick_params(colors="#CCCCCC", labelsize=8)
    ax.set_xlabel("Language (audio)", color="#CCCCCC", fontsize=10)
    ax.set_ylabel("Layer", color="#CCCCCC", fontsize=10)
    cb = ax.collections[0].colorbar
    cb.ax.yaxis.label.set_color("white"); cb.ax.tick_params(colors="white")
    plt.tight_layout()
    p2 = f"{cfg['output_dir']}/similarity_heatmap.png"
    plt.savefig(p2, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved -> {p2}")

    # zone bars
    zs = {"Acoustic\n(L1-6)": slice(1, 7),
          "Phonetic\n(L7-12)": slice(7, 13),
          "Linguistic\n(L13-24)": slice(13, 25)}
    zn = list(zs.keys()); langs = list(similarity_dict.keys())
    nl = len(langs); tw = 0.7; bw = tw / nl
    offs = np.linspace(-tw / 2 + bw / 2, tw / 2 - bw / 2, nl)
    x = np.arange(len(zn))
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(BG); ax.set_facecolor(BG)
    for idx, lang in enumerate(langs):
        sims = similarity_dict[lang]
        vals = [float(np.mean(sims[s])) for s in zs.values()]
        bars = ax.bar(x + offs[idx], vals, bw, label=lang,
                      color=_color(lang, idx), alpha=0.85, edgecolor="#222222")
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                    f"{val:.3f}", ha="center", va="bottom",
                    color="white", fontsize=8, fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(zn, color="#CCCCCC", fontsize=11)
    ax.set_ylabel("Mean Cosine Similarity", color="#CCCCCC", fontsize=11)
    ax.set_title("Zone-averaged Similarity with Bhili (Bhili model space)",
                 color="white", fontsize=13, fontweight="bold", pad=12)
    ax.tick_params(colors="#AAAAAA"); ax.spines[:].set_color("#333333")
    ax.grid(True, axis="y", alpha=0.15, color="#FFFFFF", linestyle="--")
    ax.set_ylim(0, 1.08)
    ax.legend(facecolor="#1A1A2E", edgecolor="#444444", labelcolor="white", fontsize=10)
    plt.tight_layout()
    p3 = f"{cfg['output_dir']}/zone_similarity_bars.png"
    plt.savefig(p3, dpi=180, bbox_inches="tight", facecolor=BG); plt.close()
    print(f"  Saved -> {p3}")
    return p1, p2, p3


def summary(similarity_dict, cfg):
    zs = {"CNN (L0)": slice(0, 1),
          "Acoustic (L1-6)": slice(1, 7),
          "Phonetic (L7-12)": slice(7, 13),
          "Linguistic (L13-24)": slice(13, 25),
          "Overall mean": slice(0, cfg["num_layers"])}
    rows = []
    for lang, sims in similarity_dict.items():
        row = {"Language": lang}
        for name, s in zs.items():
            row[name] = round(float(np.mean(sims[s])), 4)
        row["Peak layer"] = int(np.argmax(sims))
        row["Peak value"] = round(float(np.max(sims)), 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Language")
    sep = "=" * 78
    print(f"\n{sep}\n  SIMILARITY WITH BHILI IN BHILI FINE-TUNED SPACE\n{sep}")
    print(df.to_string()); print(sep)
    os.makedirs(cfg["output_dir"], exist_ok=True)
    csv = f"{cfg['output_dir']}/similarity_summary.csv"
    df.to_csv(csv); print(f"  Summary CSV -> {csv}")
    return df


# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    print(f"\nDevice       : {cfg['device']}")
    print(f"Bhili model  : {cfg['bhili_model']}")
    print(f"Audio langs  : {list(cfg['audio_sources'].keys())}")
    print(f"Reference    : {cfg['reference']}")

    print("\n[1/4] Loading Bhili fine-tuned model (shared encoder for all audio) ...")
    fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg["bhili_model"])
    model = Wav2Vec2Model.from_pretrained(cfg["bhili_model"])
    model.eval().to(cfg["device"])

    print("\n[2/4] Loading audio + encoding each language through the Bhili model ...")
    emb = {}
    for label, spec in cfg["audio_sources"].items():
        audio = load_audio(label, spec, cfg)
        emb[label] = extract_layerwise_embeddings(audio, cfg, fe, model, label)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    ref = cfg["reference"]
    similarity_dict = {}
    for label in cfg["audio_sources"]:
        if label == ref:
            continue
        sims = compute_layerwise_cosine(emb[ref], emb[label])
        similarity_dict[label] = sims
        print(f"     {label:9s} phonetic-zone mean (L7-12): {np.mean(sims[7:13]):.4f}")

    print("\n[3/4] Plots")
    plot_all(similarity_dict, cfg)

    print("\n[4/4] Summary")
    summary(similarity_dict, cfg)

    # Verdict
    phon = {l: float(np.mean(s[7:13])) for l, s in similarity_dict.items()}
    print("\n  Phonetic-zone (L7-12) similarity with Bhili (Bhili model space):")
    for l, v in sorted(phon.items(), key=lambda kv: -kv[1]):
        print(f"    {l:9s}: {v:.4f}")
    if "Mewari" in phon and "Gujarati" in phon:
        if phon["Mewari"] <= phon["Gujarati"] + 0.02:
            print(f"\n  -> Mewari ({phon['Mewari']:.4f}) sits at or below Gujarati "
                  f"({phon['Gujarati']:.4f}), the rejected donor. Supports exclusion.")
        else:
            print(f"\n  -> Mewari ({phon['Mewari']:.4f}) is ABOVE Gujarati "
                  f"({phon['Gujarati']:.4f}); interpret with care before using in rebuttal.")
    print(f"\n  (Paper's donor-adapted Gujarati phonetic value for reference: "
          f"{cfg['paper_gujarati_phonetic']})")
    print(f"\n  Done. Outputs in: {cfg['output_dir']}/\n")


if __name__ == "__main__":
    main()

In [ ]:
"""
Candidate-Donor Similarity in Bhili Fine-Tuned Space — CENTERED
===============================================================


Requirements:
    pip install transformers torch librosa datasets scipy matplotlib seaborn pandas numpy tqdm
"""

import os, glob, random, warnings
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import cosine
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

CONFIG = {
    "bhili_model": "model",
    "audio_sources": {
        "Bhili":    {"type": "dir",     "path": "bhili_audio"},
        "Mewari":   {"type": "parquet", "path": "mewari.parquet"},
        "Gujarati": {"type": "parquet", "path": "gujarati.parquet"},
    },
    "reference": "Bhili",
    "num_samples":    417,
    "num_layers":     25,
    "output_dir":     "mewari_centered_outputs",
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "sample_rate":    16000,
    "max_duration_s": 10,
    "seed":           42,
}

BG = "#0F1117"
ZONES = [
    (0,  1,  "#3A86FF", "CNN Features (L0)"),
    (1,  7,  "#8338EC", "Acoustic (L1-6)"),
    (7,  13, "#FF006E", "Phonetic (L7-12)"),
    (13, 25, "#FB5607", "Linguistic (L13-24)"),
]
LANG_COLORS = {"Mewari": "#6A4C93", "Gujarati": "#E63946"}


# =============================================================================
# Audio loading
# =============================================================================

def _finalize(arr, cfg):
    max_len = int(cfg["max_duration_s"] * cfg["sample_rate"])
    arr = np.asarray(arr, dtype=np.float32)[:max_len]
    if len(arr) < 400:
        return None
    peak = np.abs(arr).max()
    if peak > 0:
        arr = arr / peak
    return arr


def _load_dir(path, cfg, label):
    exts = ("*.wav", "*.flac", "*.mp3", "*.ogg")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(path, "**", ext), recursive=True))
    files = sorted(files)
    if not files:
        raise FileNotFoundError(f"No audio under '{path}' for {label}.")
    random.seed(cfg["seed"])
    if len(files) > cfg["num_samples"]:
        files = random.sample(files, cfg["num_samples"])
    out, skipped = [], 0
    for fp in tqdm(files, desc=f"  {label} (dir)", leave=False):
        try:
            wav, _ = librosa.load(fp, sr=cfg["sample_rate"], mono=True)
            wav = _finalize(wav, cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {label}: loaded {len(out)} clips ({skipped} skipped)")
    return out


def _load_parquet(path, cfg, label):
    from datasets import load_dataset, Audio
    ds = load_dataset("parquet", data_files=path)
    ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
    ds = ds.cast_column("audio", Audio(sampling_rate=cfg["sample_rate"]))
    idxs = list(range(len(ds)))
    random.seed(cfg["seed"])
    if len(idxs) > cfg["num_samples"]:
        idxs = random.sample(idxs, cfg["num_samples"])
    out, skipped = [], 0
    for i in tqdm(idxs, desc=f"  {label} (parquet)", leave=False):
        try:
            wav = _finalize(ds[i]["audio"]["array"], cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {label}: loaded {len(out)} clips ({skipped} skipped)")
    return out


def load_audio(label, spec, cfg):
    if spec["type"] == "dir":
        return _load_dir(spec["path"], cfg, label)
    if spec["type"] == "parquet":
        return _load_parquet(spec["path"], cfg, label)
    raise ValueError(f"Unknown source type: {spec['type']}")


# =============================================================================
# Embeddings + centering + similarity
# =============================================================================

def extract_layerwise_embeddings(audio_list, cfg, fe, model, label):
    device = cfg["device"]
    layer_accum, count = None, 0
    for audio in tqdm(audio_list, desc=f"    Encoding {label}", leave=False):
        try:
            inputs = fe(audio, sampling_rate=cfg["sample_rate"],
                        return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)
            clip = np.stack([hs.squeeze(0).mean(0).cpu().float().numpy()
                             for hs in out.hidden_states])
            layer_accum = clip if layer_accum is None else layer_accum + clip
            count += 1
        except Exception:
            pass
    if count == 0:
        raise RuntimeError(f"No embeddings extracted for {label}.")
    return layer_accum / count       # (num_layers, D)


def center_embeddings(emb_dict, num_layers):
    """Subtract, per layer, the across-language mean embedding.
    emb_dict: {lang -> (num_layers, D)}. Returns a new dict, same shapes."""
    langs = list(emb_dict.keys())
    stacked = np.stack([emb_dict[l] for l in langs])   # (L_langs, num_layers, D)
    mean_per_layer = stacked.mean(axis=0)              # (num_layers, D)
    return {l: emb_dict[l] - mean_per_layer for l in langs}


def compute_layerwise_cosine(ref, other):
    return [float(1 - cosine(ref[i], other[i])) for i in range(ref.shape[0])]


# =============================================================================
# Summary helpers
# =============================================================================

def summarize(similarity_dict, cfg, tag):
    zs = {"CNN (L0)": slice(0, 1),
          "Acoustic (L1-6)": slice(1, 7),
          "Phonetic (L7-12)": slice(7, 13),
          "Linguistic (L13-24)": slice(13, 25),
          "Overall mean": slice(0, cfg["num_layers"])}
    rows = []
    for lang, sims in similarity_dict.items():
        sims = np.asarray(sims)
        row = {"Language": lang}
        for name, s in zs.items():
            row[name] = round(float(np.mean(sims[s])), 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Language")
    sep = "=" * 78
    print(f"\n{sep}\n  [{tag}] SIMILARITY WITH BHILI  (Bhili fine-tuned space)\n{sep}")
    print(df.to_string()); print(sep)
    os.makedirs(cfg["output_dir"], exist_ok=True)
    df.to_csv(f"{cfg['output_dir']}/similarity_{tag}.csv")
    return df


def verdict(similarity_dict, tag):
    phon = {l: float(np.mean(np.asarray(s)[7:13])) for l, s in similarity_dict.items()}
    print(f"\n  [{tag}] Phonetic-zone (L7-12) similarity with Bhili:")
    for l, v in sorted(phon.items(), key=lambda kv: -kv[1]):
        print(f"    {l:9s}: {v:.4f}")
    if "Mewari" in phon and "Gujarati" in phon:
        gap = phon["Gujarati"] - phon["Mewari"]
        rel = ("at or below" if phon["Mewari"] <= phon["Gujarati"] else "above")
        print(f"    Mewari is {rel} Gujarati (gap = {gap:+.4f}).")
    return phon


# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    print(f"\nDevice      : {cfg['device']}")
    print(f"Bhili model : {cfg['bhili_model']}")
    print(f"Audio langs : {list(cfg['audio_sources'].keys())}")

    print("\n[1/4] Loading Bhili fine-tuned model ...")
    fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg["bhili_model"])
    model = Wav2Vec2Model.from_pretrained(cfg["bhili_model"])
    model.eval().to(cfg["device"])

    print("\n[2/4] Encoding each language through the Bhili model ...")
    emb = {}
    for label, spec in cfg["audio_sources"].items():
        audio = load_audio(label, spec, cfg)
        emb[label] = extract_layerwise_embeddings(audio, cfg, fe, model, label)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    ref = cfg["reference"]

    # ---- RAW (anisotropic, for reference only) ----
    raw = {l: compute_layerwise_cosine(emb[ref], emb[l])
           for l in emb if l != ref}
    summarize(raw, cfg, "RAW")
    verdict(raw, "RAW")

    # ---- CENTERED (common component removed; the real comparison) ----
    emb_c = center_embeddings(emb, cfg["num_layers"])
    cen = {l: compute_layerwise_cosine(emb_c[ref], emb_c[l])
           for l in emb_c if l != ref}
    summarize(cen, cfg, "CENTERED")
    phon = verdict(cen, "CENTERED")

    print("\n  NOTE: centered cosines are on their own internal scale (can be")
    print("  negative) and are NOT comparable to the paper's 0.067 / 0.751 values.")
    print("  Use them only for the relative Mewari-vs-Gujarati claim.\n")

    if "Mewari" in phon and "Gujarati" in phon:
        if phon["Mewari"] <= phon["Gujarati"]:
            print("  => CENTERED result supports exclusion: Mewari separates from")
            print("     Bhili at least as much as the rejected Gujarati donor.\n")
        else:
            print("  => CENTERED result does NOT cleanly support exclusion; Mewari")
            print("     is closer to Bhili than Gujarati. Do not use numeric claim.\n")

    print(f"  Done. Outputs in: {cfg['output_dir']}/\n")


if __name__ == "__main__":
    main()

In [ ]:
"""
Candidate-Donor Similarity in Bhili Fine-Tuned Space — CENTERED
===============================================================


Requirements:
    pip install transformers torch librosa datasets scipy matplotlib seaborn pandas numpy tqdm
"""

import os, glob, random, warnings
import numpy as np
import torch
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from tqdm import tqdm
from scipy.spatial.distance import cosine
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIGURATION
# =============================================================================

CONFIG = {
    "bhili_model": "model",
    "audio_sources": {
        "Bhili":    {"type": "dir",     "path": "bhili_audio"},
        "Mewari":   {"type": "parquet", "path": "mewari.parquet"},
        "Gujarati": {"type": "parquet", "path": "gujarati.parquet"},
    },
    "reference": "Bhili",
    "num_samples":    417,
    "num_layers":     25,
    "output_dir":     "mewari_centered_outputs",
    "device":         "cuda" if torch.cuda.is_available() else "cpu",
    "sample_rate":    16000,
    "max_duration_s": 10,
    "seed":           42,
}

BG = "#0F1117"
ZONES = [
    (0,  1,  "#3A86FF", "CNN Features (L0)"),
    (1,  7,  "#8338EC", "Acoustic (L1-6)"),
    (7,  13, "#FF006E", "Phonetic (L7-12)"),
    (13, 25, "#FB5607", "Linguistic (L13-24)"),
]
LANG_COLORS = {"Mewari": "#6A4C93", "Gujarati": "#E63946"}


# =============================================================================
# Audio loading
# =============================================================================

def _finalize(arr, cfg):
    max_len = int(cfg["max_duration_s"] * cfg["sample_rate"])
    arr = np.asarray(arr, dtype=np.float32)[:max_len]
    if len(arr) < 400:
        return None
    peak = np.abs(arr).max()
    if peak > 0:
        arr = arr / peak
    return arr


def _load_dir(path, cfg, label):
    exts = ("*.wav", "*.flac", "*.mp3", "*.ogg")
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(path, "**", ext), recursive=True))
    files = sorted(files)
    if not files:
        raise FileNotFoundError(f"No audio under '{path}' for {label}.")
    random.seed(cfg["seed"])
    if len(files) > cfg["num_samples"]:
        files = random.sample(files, cfg["num_samples"])
    out, skipped = [], 0
    for fp in tqdm(files, desc=f"  {label} (dir)", leave=False):
        try:
            wav, _ = librosa.load(fp, sr=cfg["sample_rate"], mono=True)
            wav = _finalize(wav, cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {label}: loaded {len(out)} clips ({skipped} skipped)")
    return out


def _load_parquet(path, cfg, label):
    from datasets import load_dataset, Audio
    ds = load_dataset("parquet", data_files=path)
    ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
    ds = ds.cast_column("audio", Audio(sampling_rate=cfg["sample_rate"]))
    idxs = list(range(len(ds)))
    random.seed(cfg["seed"])
    if len(idxs) > cfg["num_samples"]:
        idxs = random.sample(idxs, cfg["num_samples"])
    out, skipped = [], 0
    for i in tqdm(idxs, desc=f"  {label} (parquet)", leave=False):
        try:
            wav = _finalize(ds[i]["audio"]["array"], cfg)
            if wav is None:
                skipped += 1; continue
            out.append(wav)
        except Exception:
            skipped += 1
    print(f"  {label}: loaded {len(out)} clips ({skipped} skipped)")
    return out


def load_audio(label, spec, cfg):
    if spec["type"] == "dir":
        return _load_dir(spec["path"], cfg, label)
    if spec["type"] == "parquet":
        return _load_parquet(spec["path"], cfg, label)
    raise ValueError(f"Unknown source type: {spec['type']}")


# =============================================================================
# Embeddings + centering + similarity
# =============================================================================

def extract_layerwise_embeddings(audio_list, cfg, fe, model, label):
    device = cfg["device"]
    layer_accum, count = None, 0
    for audio in tqdm(audio_list, desc=f"    Encoding {label}", leave=False):
        try:
            inputs = fe(audio, sampling_rate=cfg["sample_rate"],
                        return_tensors="pt").to(device)
            with torch.no_grad():
                out = model(**inputs, output_hidden_states=True)
            clip = np.stack([hs.squeeze(0).mean(0).cpu().float().numpy()
                             for hs in out.hidden_states])
            layer_accum = clip if layer_accum is None else layer_accum + clip
            count += 1
        except Exception:
            pass
    if count == 0:
        raise RuntimeError(f"No embeddings extracted for {label}.")
    return layer_accum / count       # (num_layers, D)


def center_embeddings(emb_dict, num_layers):
    """Subtract, per layer, the across-language mean embedding.
    emb_dict: {lang -> (num_layers, D)}. Returns a new dict, same shapes."""
    langs = list(emb_dict.keys())
    stacked = np.stack([emb_dict[l] for l in langs])   # (L_langs, num_layers, D)
    mean_per_layer = stacked.mean(axis=0)              # (num_layers, D)
    return {l: emb_dict[l] - mean_per_layer for l in langs}


def compute_layerwise_cosine(ref, other):
    return [float(1 - cosine(ref[i], other[i])) for i in range(ref.shape[0])]


# =============================================================================
# Summary helpers
# =============================================================================

def summarize(similarity_dict, cfg, tag):
    zs = {"CNN (L0)": slice(0, 1),
          "Acoustic (L1-6)": slice(1, 7),
          "Phonetic (L7-12)": slice(7, 13),
          "Linguistic (L13-24)": slice(13, 25),
          "Overall mean": slice(0, cfg["num_layers"])}
    rows = []
    for lang, sims in similarity_dict.items():
        sims = np.asarray(sims)
        row = {"Language": lang}
        for name, s in zs.items():
            row[name] = round(float(np.mean(sims[s])), 4)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("Language")
    sep = "=" * 78
    print(f"\n{sep}\n  [{tag}] SIMILARITY WITH BHILI  (Bhili fine-tuned space)\n{sep}")
    print(df.to_string()); print(sep)
    os.makedirs(cfg["output_dir"], exist_ok=True)
    df.to_csv(f"{cfg['output_dir']}/similarity_{tag}.csv")
    return df


def verdict(similarity_dict, tag):
    phon = {l: float(np.mean(np.asarray(s)[7:13])) for l, s in similarity_dict.items()}
    print(f"\n  [{tag}] Phonetic-zone (L7-12) similarity with Bhili:")
    for l, v in sorted(phon.items(), key=lambda kv: -kv[1]):
        print(f"    {l:9s}: {v:.4f}")
    if "Mewari" in phon and "Gujarati" in phon:
        gap = phon["Gujarati"] - phon["Mewari"]
        rel = ("at or below" if phon["Mewari"] <= phon["Gujarati"] else "above")
        print(f"    Mewari is {rel} Gujarati (gap = {gap:+.4f}).")
    return phon


# =============================================================================
# MAIN
# =============================================================================

def main():
    cfg = CONFIG
    print(f"\nDevice      : {cfg['device']}")
    print(f"Bhili model : {cfg['bhili_model']}")
    print(f"Audio langs : {list(cfg['audio_sources'].keys())}")

    print("\n[1/4] Loading Bhili fine-tuned model ...")
    fe = Wav2Vec2FeatureExtractor.from_pretrained(cfg["bhili_model"])
    model = Wav2Vec2Model.from_pretrained(cfg["bhili_model"])
    model.eval().to(cfg["device"])

    print("\n[2/4] Encoding each language through the Bhili model ...")
    emb = {}
    for label, spec in cfg["audio_sources"].items():
        audio = load_audio(label, spec, cfg)
        emb[label] = extract_layerwise_embeddings(audio, cfg, fe, model, label)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    ref = cfg["reference"]

    # ---- RAW (anisotropic, for reference only) ----
    raw = {l: compute_layerwise_cosine(emb[ref], emb[l])
           for l in emb if l != ref}
    summarize(raw, cfg, "RAW")
    verdict(raw, "RAW")

    # ---- CENTERED (common component removed; the real comparison) ----
    emb_c = center_embeddings(emb, cfg["num_layers"])
    cen = {l: compute_layerwise_cosine(emb_c[ref], emb_c[l])
           for l in emb_c if l != ref}
    summarize(cen, cfg, "CENTERED")
    phon = verdict(cen, "CENTERED")

    print("\n  NOTE: centered cosines are on their own internal scale (can be")
    print("  negative) and are NOT comparable to the paper's 0.067 / 0.751 values.")
    print("  Use them only for the relative Mewari-vs-Gujarati claim.\n")

    if "Mewari" in phon and "Gujarati" in phon:
        if phon["Mewari"] <= phon["Gujarati"]:
            print("  => CENTERED result supports exclusion: Mewari separates from")
            print("     Bhili at least as much as the rejected Gujarati donor.\n")
        else:
            print("  => CENTERED result does NOT cleanly support exclusion; Mewari")
            print("     is closer to Bhili than Gujarati. Do not use numeric claim.\n")

    print(f"  Done. Outputs in: {cfg['output_dir']}/\n")


if __name__ == "__main__":
    main()